## 1. `requirements.txt` and install

In [1]:
%%writefile requirements.txt
fastapi
uvicorn
scikit-learn
pandas
joblib


Writing requirements.txt


In [2]:
# Install packages. httpx is used by the API test cell.
%pip install -q fastapi uvicorn scikit-learn pandas joblib httpx

## 2. Project folders

In [3]:

from pathlib import Path
import pandas as pd

# Create folders if they do not exist
for d in ["data", "models", "frontend"]:
    Path(d).mkdir(exist_ok=True)

# Read the CSV file
DATA_PATH = Path("data/cleaned_LOAN.csv")

# If the file is in the notebook folder, use that path instead
if not DATA_PATH.exists():
    DATA_PATH = Path("cleaned_LOAN.csv")

# Load the dataset using pandas
# Note: Avoid using Windows local paths like C:\... in Colab as they do not exist in this environment.
if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)

    # Display the first 5 rows
    print(df.head())

    # Check the dataset size
    print("Dataset shape:", df.shape)

    # Check column names
    print(df.columns)

    # Display the dataset
    print("Dataset loaded successfully!")
    print("File location:", DATA_PATH)
else:
    print(f"Please upload '{DATA_PATH.name}' ")


   education  self_employed  income_annum  loan_amount  loan_term  \
0          1              0       9600000     29900000         12   
1          0              1       4100000     12200000          8   
2          1              0       9100000     29700000         20   
3          1              0       8200000     30700000          8   
4          0              1       9800000     24200000         20   

   cibil_score  bank_asset_value  loan_status  
0          778           8000000            1  
1          417           3300000            0  
2          506          12800000            0  
3          467           7900000            0  
4          382           5000000            0  
Dataset shape: (4269, 8)
Index(['education', 'self_employed', 'income_annum', 'loan_amount',
       'loan_term', 'cibil_score', 'bank_asset_value', 'loan_status'],
      dtype='object')
Dataset loaded successfully!
File location: cleaned_LOAN.csv


## 3. Explore the data

In [4]:
import pandas as pd

df = pd.read_csv(DATA_PATH)
print(df.shape)
display(df.head())
display(df.describe().T)
print(df['loan_status'].value_counts(normalize=True))

(4269, 8)


,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,bank_asset_value,loan_status
0,1,0,9600000,29900000,12,778,8000000,1
1,0,1,4100000,12200000,8,417,3300000,0
2,1,0,9100000,29700000,20,506,12800000,0
3,1,0,8200000,30700000,8,467,7900000,0
4,0,1,9800000,24200000,20,382,5000000,0


,count,mean,std,min,25%,50%,75%,max
education,4269.0,5.022253e-01,5.000536e-01,0.0,0.0,1.0,1.0,1.0
self_employed,4269.0,5.036308e-01,5.000454e-01,0.0,0.0,1.0,1.0,1.0
income_annum,4269.0,5.059124e+06,2.806840e+06,200000.0,2700000.0,5100000.0,7500000.0,9900000.0
loan_amount,4269.0,1.513345e+07,9.043363e+06,300000.0,7700000.0,14500000.0,21500000.0,39500000.0
loan_term,4269.0,1.090045e+01,5.709187e+00,2.0,6.0,10.0,16.0,20.0
cibil_score,4269.0,5.999361e+02,1.724304e+02,300.0,453.0,600.0,748.0,900.0
bank_asset_value,4269.0,4.976692e+06,3.250185e+06,0.0,2300000.0,4600000.0,7100000.0,14700000.0
loan_status,4269.0,6.221598e-01,4.849042e-01,0.0,0.0,1.0,1.0,1.0


loan_status
1    0.62216
0    0.37784
Name: proportion, dtype: float64


## 4. `train.py` – train, evaluate and save both models

In [5]:
%%writefile train.py
"""Train a regression model and a classification model on the loan dataset.

Regression     : predict loan_amount (a number) with LinearRegression
Classification : predict loan_status (0 = rejected, 1 = approved) with LogisticRegression

Run:  python train.py
"""
import joblib
import pandas as pd
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import accuracy_score, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("data/cleaned_LOAN.csv")

# ---------- 1. Regression: predict loan_amount ----------
REG_FEATURES = [
    "education", "self_employed", "income_annum",
    "loan_term", "cibil_score", "bank_asset_value",
]
X = df[REG_FEATURES]
y = df["loan_amount"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

reg_model = LinearRegression()
reg_model.fit(X_train, y_train)
r2 = r2_score(y_test, reg_model.predict(X_test))
print(f"Regression     R2       : {r2:.4f}")

# ---------- 2. Classification: predict loan_status ----------
CLF_FEATURES = [
    "education", "self_employed", "income_annum", "loan_amount",
    "loan_term", "cibil_score", "bank_asset_value",
]
X = df[CLF_FEATURES]
y = df["loan_status"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scaling matters here: income/loan values are in the millions, cibil_score is ~300-900.
clf_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
clf_model.fit(X_train, y_train)
acc = accuracy_score(y_test, clf_model.predict(X_test))
print(f"Classification accuracy : {acc:.4f}")

# ---------- 3. Save ----------
joblib.dump({"model": reg_model, "features": REG_FEATURES}, "models/regression_model.joblib")
joblib.dump({"model": clf_model, "features": CLF_FEATURES}, "models/classification_model.joblib")
print("Saved models to models/")


Writing train.py


In [6]:
# Same logic as train.py, run inline so results show in the notebook
import joblib
import pandas as pd
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import accuracy_score, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv(DATA_PATH)

# ---------- 1. Regression: predict loan_amount ----------
REG_FEATURES = [
    "education", "self_employed", "income_annum",
    "loan_term", "cibil_score", "bank_asset_value",
]
X = df[REG_FEATURES]
y = df["loan_amount"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

reg_model = LinearRegression()
reg_model.fit(X_train, y_train)
r2 = r2_score(y_test, reg_model.predict(X_test))
print(f"Regression     R2       : {r2:.4f}")

# ---------- 2. Classification: predict loan_status ----------
CLF_FEATURES = [
    "education", "self_employed", "income_annum", "loan_amount",
    "loan_term", "cibil_score", "bank_asset_value",
]
X = df[CLF_FEATURES]
y = df["loan_status"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scaling matters here: income/loan values are in the millions, cibil_score is ~300-900.
clf_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
clf_model.fit(X_train, y_train)
acc = accuracy_score(y_test, clf_model.predict(X_test))
print(f"Classification accuracy : {acc:.4f}")

# ---------- 3. Save ----------
joblib.dump({"model": reg_model, "features": REG_FEATURES}, "models/regression_model.joblib")
joblib.dump({"model": clf_model, "features": CLF_FEATURES}, "models/classification_model.joblib")
print("Saved models to models/")


Regression     R2       : 0.8513
Classification accuracy : 0.9122
Saved models to models/


## 5. `main.py` – FastAPI backend

In [7]:
%%writefile main.py
"""FastAPI backend: loads both saved models and exposes two POST endpoints.

Run:  uvicorn main:app --reload
"""
from pathlib import Path

import joblib
import pandas as pd
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse
from pydantic import BaseModel, Field

BASE_DIR = Path(__file__).parent

# Load models once at startup
reg_bundle = joblib.load(BASE_DIR / "models" / "regression_model.joblib")
clf_bundle = joblib.load(BASE_DIR / "models" / "classification_model.joblib")

app = FastAPI(title="Loan ML App")

# Allows the HTML page to work even if opened as a file or served from elsewhere
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])


class RegressionInput(BaseModel):
    education: int = Field(..., ge=0, le=1)
    self_employed: int = Field(..., ge=0, le=1)
    income_annum: float = Field(..., ge=0)
    loan_term: int = Field(..., ge=1)
    cibil_score: int = Field(..., ge=300, le=900)
    bank_asset_value: float = Field(..., ge=0)


class ClassificationInput(RegressionInput):
    loan_amount: float = Field(..., ge=0)


@app.get("/")
def index():
    """Serve the frontend page."""
    return FileResponse(BASE_DIR / "frontend" / "index.html")


@app.post("/predict/regression")
def predict_regression(data: RegressionInput):
    row = pd.DataFrame([data.model_dump()])[reg_bundle["features"]]
    prediction = float(reg_bundle["model"].predict(row)[0])
    return {"predicted_loan_amount": round(max(prediction, 0), 2)}


@app.post("/predict/classification")
def predict_classification(data: ClassificationInput):
    row = pd.DataFrame([data.model_dump()])[clf_bundle["features"]]
    model = clf_bundle["model"]
    label = int(model.predict(row)[0])
    probability = float(model.predict_proba(row)[0][label])
    return {
        "loan_status": label,
        "label": "Approved" if label == 1 else "Rejected",
        "confidence": round(probability, 4),
    }


Writing main.py


## 6. `frontend/index.html` – web page

In [8]:
%%writefile frontend/index.html
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>Loan ML App</title>
  <style>
    body { font-family: system-ui, sans-serif; background: #f4f6f8; margin: 0; padding: 24px; color: #222; }
    h1 { text-align: center; margin-bottom: 4px; }
    .sub { text-align: center; color: #666; margin-bottom: 24px; }
    .wrap { display: flex; flex-wrap: wrap; gap: 24px; justify-content: center; }
    .card { background: #fff; border-radius: 10px; padding: 20px; width: 360px; box-shadow: 0 2px 8px rgba(0,0,0,.08); }
    h2 { margin-top: 0; font-size: 1.15rem; }
    label { display: block; margin: 10px 0 4px; font-size: .9rem; }
    input, select { width: 100%; padding: 8px; box-sizing: border-box; border: 1px solid #ccc; border-radius: 6px; }
    button { margin-top: 16px; width: 100%; padding: 10px; border: 0; border-radius: 6px; background: #2563eb; color: #fff; font-size: 1rem; cursor: pointer; }
    button:hover { background: #1d4ed8; }
    .result { margin-top: 16px; padding: 12px; border-radius: 6px; background: #eef2ff; min-height: 1.2em; }
    .error { background: #fee2e2; color: #991b1b; }
  </style>
</head>
<body>
  <h1>Loan ML App</h1>
  <p class="sub">Linear regression predicts a loan amount. Logistic regression predicts approval.</p>

  <div class="wrap">
    <!-- Regression form -->
    <div class="card">
      <h2>Regression: Predict Loan Amount</h2>
      <form id="reg-form">
        <label>Education</label>
        <select name="education"><option value="1">Graduate</option><option value="0">Not Graduate</option></select>
        <label>Self employed</label>
        <select name="self_employed"><option value="0">No</option><option value="1">Yes</option></select>
        <label>Annual income</label>
        <input type="number" name="income_annum" value="5000000" min="0" required>
        <label>Loan term (years)</label>
        <input type="number" name="loan_term" value="10" min="1" required>
        <label>CIBIL score (300-900)</label>
        <input type="number" name="cibil_score" value="650" min="300" max="900" required>
        <label>Bank asset value</label>
        <input type="number" name="bank_asset_value" value="5000000" min="0" required>
        <button type="submit">Predict amount</button>
      </form>
      <div class="result" id="reg-result"></div>
    </div>

    <!-- Classification form -->
    <div class="card">
      <h2>Classification: Loan Approval</h2>
      <form id="clf-form">
        <label>Education</label>
        <select name="education"><option value="1">Graduate</option><option value="0">Not Graduate</option></select>
        <label>Self employed</label>
        <select name="self_employed"><option value="0">No</option><option value="1">Yes</option></select>
        <label>Annual income</label>
        <input type="number" name="income_annum" value="5000000" min="0" required>
        <label>Loan amount</label>
        <input type="number" name="loan_amount" value="15000000" min="0" required>
        <label>Loan term (years)</label>
        <input type="number" name="loan_term" value="10" min="1" required>
        <label>CIBIL score (300-900)</label>
        <input type="number" name="cibil_score" value="650" min="300" max="900" required>
        <label>Bank asset value</label>
        <input type="number" name="bank_asset_value" value="5000000" min="0" required>
        <button type="submit">Predict status</button>
      </form>
      <div class="result" id="clf-result"></div>
    </div>
  </div>

  <script>
    // If the page is opened straight from disk, talk to the local server; otherwise use the same origin.
    const API = location.protocol === "file:" ? "http://127.0.0.1:8000" : "";

    async function send(form, endpoint, resultEl, render) {
      const out = document.getElementById(resultEl);
      out.className = "result";
      out.textContent = "Predicting...";
      // Convert every form field to a number
      const body = Object.fromEntries(
        [...new FormData(form).entries()].map(([k, v]) => [k, Number(v)])
      );
      try {
        const res = await fetch(API + endpoint, {
          method: "POST",
          headers: { "Content-Type": "application/json" },
          body: JSON.stringify(body),
        });
        if (!res.ok) throw new Error("Server returned " + res.status);
        out.textContent = render(await res.json());
      } catch (err) {
        out.className = "result error";
        out.textContent = "Error: " + err.message;
      }
    }

    document.getElementById("reg-form").addEventListener("submit", (e) => {
      e.preventDefault();
      send(e.target, "/predict/regression", "reg-result",
        (d) => "Predicted loan amount: " + d.predicted_loan_amount.toLocaleString());
    });

    document.getElementById("clf-form").addEventListener("submit", (e) => {
      e.preventDefault();
      send(e.target, "/predict/classification", "clf-result",
        (d) => d.label + " (confidence " + (d.confidence * 100).toFixed(1) + "%)");
    });
  </script>
</body>
</html>


Writing frontend/index.html


## 7. Test the API inside the notebook

In [9]:
from fastapi.testclient import TestClient
import main

client = TestClient(main.app)

reg = client.post("/predict/regression", json={
    "education": 1, "self_employed": 0, "income_annum": 5000000,
    "loan_term": 10, "cibil_score": 650, "bank_asset_value": 5000000})
print(reg.status_code, reg.json())

clf = client.post("/predict/classification", json={
    "education": 1, "self_employed": 0, "income_annum": 5000000, "loan_amount": 15000000,
    "loan_term": 10, "cibil_score": 650, "bank_asset_value": 5000000})
print(clf.status_code, clf.json())

200 {'predicted_loan_amount': 14953902.29}
200 {'loan_status': 1, 'label': 'Approved', 'confidence': 0.9576}


## 8. Run the web app

In [10]:
# Start the web server in a background thread (the notebook stays usable)
import threading, time, uvicorn
import main

config = uvicorn.Config(main.app, host="0.0.0.0", port=8000, log_level="warning")
server = uvicorn.Server(config)
thread = threading.Thread(target=server.run, daemon=True)
thread.start()
time.sleep(2)

try:
    # ---- Google Colab: show the page inside the notebook ----
    from google.colab import output
    print("(If the frame is blank, open the link printed here in a new tab.)")
    print("Public proxy URL:", output.eval_js("google.colab.kernel.proxyPort(8000)"))
    output.serve_kernel_port_as_iframe(8000, height=700)
except ImportError:
    # ---- Local Jupyter ----
    print("Server running -> open http://127.0.0.1:8000 in your browser")
# To stop it later, run:  server.should_exit = True

(If the frame is blank, open the link printed here in a new tab.)
Public proxy URL: https://8000-m-s-usw3b8-tuzwrr6ye7tr-b.us-west3.prod.r.colab.dev


<IPython.core.display.Javascript object>